In [0]:
-- Q01: Overall KPI Summary
SELECT 
    SUM(extendedprice * (1 - discount)) AS net_revenue,
    COUNT(DISTINCT orderkey) AS orders,
    COUNT(DISTINCT customerkey) AS active_customers,
    SUM(extendedprice * (1 - discount)) / COUNT(DISTINCT orderkey) AS avg_order_value
FROM dev.gold.fact_sales;

In [0]:
-- Q02: Monthly Revenue Trend Across All Years
SELECT 
    YEAR(orderdate) AS year,
    MONTH(orderdate) AS month,
    SUM(extendedprice * (1 - discount)) AS monthly_revenue,
    COUNT(DISTINCT orderkey) AS orders,
    COUNT(DISTINCT customerkey) AS active_customers
FROM dev.gold.fact_sales
GROUP BY year, month
ORDER BY year, month;

In [0]:
-- Q03: Annual Revenue with Year-Over-Year Change
SELECT 
    YEAR(orderdate) AS year,
    SUM(extendedprice * (1 - discount)) AS annual_revenue,
    COUNT(DISTINCT orderkey) AS orders,
    COUNT(DISTINCT customerkey) AS active_customers,
    SUM(extendedprice * (1 - discount)) / COUNT(DISTINCT orderkey) AS avg_order_value,
    LAG(SUM(extendedprice * (1 - discount))) OVER (ORDER BY YEAR(orderdate)) AS prior_year_revenue,
    (SUM(extendedprice * (1 - discount)) - LAG(SUM(extendedprice * (1 - discount))) OVER (ORDER BY YEAR(orderdate))) / NULLIF(LAG(SUM(extendedprice * (1 - discount))) OVER (ORDER BY YEAR(orderdate)),0) AS yoy_change
FROM dev.gold.fact_sales
GROUP BY year
ORDER BY year;

In [0]:
-- Q04: Revenue by Customer Market Segment
SELECT dc.market_segment, 
    SUM(fs.extendedprice * (1 - fs.discount)) AS segment_revenue,
    COUNT(DISTINCT fs.orderkey) AS orders,
    COUNT(DISTINCT fs.customerkey) AS active_customers,
    SUM(fs.extendedprice * (1 - fs.discount)) / COUNT(DISTINCT fs.orderkey) AS avg_order_value
FROM dev.gold.fact_sales fs
JOIN dev.gold.dim_customer dc ON fs.customerkey = dc.customerkey
GROUP BY dc.market_segment
ORDER BY segment_revenue DESC;

In [0]:
-- Q05: Top 10 Customers by Net Revenue
SELECT dc.customer_name,
    SUM(fs.extendedprice * (1 - fs.discount)) AS net_revenue,
    COUNT(DISTINCT fs.orderkey) AS orders,
    SUM(fs.extendedprice * (1 - fs.discount))/COUNT(DISTINCT fs.orderkey) AS avg_order_value
FROM dev.gold.fact_sales fs
JOIN dev.gold.dim_customer dc ON fs.customerkey = dc.customerkey
GROUP BY dc.customer_name
ORDER BY net_revenue DESC
LIMIT 10;

In [0]:
-- Q06: Top 10 Customers as a Share of Total Revenue
WITH top_customers AS (
    SELECT dc.customer_name, SUM(fs.extendedprice * (1 - fs.discount)) AS net_revenue
    FROM dev.gold.fact_sales fs
    JOIN dev.gold.dim_customer dc ON fs.customerkey = dc.customerkey
    GROUP BY dc.customer_name
    ORDER BY net_revenue DESC
    LIMIT 10
), total AS (
    SELECT SUM(extendedprice * (1 - discount)) AS total_revenue FROM dev.gold.fact_sales
)
SELECT (SELECT SUM(net_revenue) FROM top_customers) / total.total_revenue AS top10_share
FROM total;

In [0]:
-- Q07: Top 10 Products by Net Revenue
SELECT dp.product_name,
    SUM(fs.extendedprice * (1 - fs.discount)) AS net_revenue,
    COUNT(DISTINCT fs.orderkey) AS orders,
    SUM(fs.extendedprice * (1 - fs.discount))/COUNT(DISTINCT fs.orderkey) AS avg_order_value
FROM dev.gold.fact_sales fs
JOIN dev.gold.dim_product dp ON fs.productkey = dp.productkey
GROUP BY dp.product_name
ORDER BY net_revenue DESC
LIMIT 10;

In [0]:
-- Q08: Bottom 10 Products by Net Revenue
SELECT dp.product_name,
    SUM(fs.extendedprice * (1 - fs.discount)) AS net_revenue,
    COUNT(DISTINCT fs.orderkey) AS orders,
    SUM(fs.extendedprice * (1 - fs.discount))/COUNT(DISTINCT fs.orderkey) AS avg_order_value
FROM dev.gold.fact_sales fs
JOIN dev.gold.dim_product dp ON fs.productkey = dp.productkey
GROUP BY dp.product_name
ORDER BY net_revenue ASC
LIMIT 10;

In [0]:
-- Q09: Revenue by Product Type
SELECT dp.type AS product_type,
    SUM(fs.extendedprice * (1 - fs.discount)) AS net_revenue,
    COUNT(DISTINCT fs.orderkey) AS orders,
    COUNT(DISTINCT fs.customerkey) AS active_customers,
    SUM(fs.extendedprice * (1 - fs.discount)) / COUNT(DISTINCT fs.orderkey) AS avg_order_value
FROM dev.gold.fact_sales fs
JOIN dev.gold.dim_product dp ON fs.productkey = dp.productkey
GROUP BY dp.type
ORDER BY net_revenue DESC;

In [0]:
-- Q10: Top 10 Suppliers by Net Revenue
SELECT fs.supplier_name,
    SUM(fs.extendedprice * (1 - fs.discount)) AS net_revenue,
    COUNT(DISTINCT fs.orderkey) AS orders,
    SUM(fs.extendedprice * (1 - fs.discount))/COUNT(DISTINCT fs.orderkey) AS avg_order_value
FROM dev.gold.fact_sales fs
GROUP BY fs.supplier_name
ORDER BY net_revenue DESC
LIMIT 10;

In [0]:
-- Q11: Revenue Share of the Top 10 Suppliers
WITH top_suppliers AS (
    SELECT supplier_name, SUM(extendedprice * (1 - discount)) AS net_revenue
    FROM dev.gold.fact_sales
    GROUP BY supplier_name
    ORDER BY net_revenue DESC
    LIMIT 10
), total AS (
    SELECT SUM(extendedprice * (1 - discount)) AS total_revenue FROM dev.gold.fact_sales
)
SELECT (SELECT SUM(net_revenue) FROM top_suppliers) / total.total_revenue AS top_10_supplier_share
FROM total;

In [0]:
-- Q12: Revenue by Customer Region
SELECT fs.region_name AS customer_region,
    SUM(fs.extendedprice * (1 - fs.discount)) AS net_revenue,
    COUNT(DISTINCT fs.orderkey) AS orders,
    COUNT(DISTINCT fs.customerkey) AS active_customers
FROM dev.gold.fact_sales fs
GROUP BY fs.region_name
ORDER BY net_revenue DESC;

In [0]:
-- Q13: Revenue by Customer Nation (All Nations)
SELECT fs.nation_name AS customer_nation,
    SUM(fs.extendedprice * (1 - fs.discount)) AS net_revenue,
    COUNT(DISTINCT fs.orderkey) AS orders,
    COUNT(DISTINCT fs.customerkey) AS active_customers
FROM dev.gold.fact_sales fs
GROUP BY fs.nation_name
ORDER BY net_revenue DESC;

In [0]:
-- Q14: Revenue and Volume Split by Return Flag
SELECT l_returnflag,
    SUM(extendedprice * (1 - discount)) AS net_revenue,
    SUM(quantity) AS total_quantity,
    COUNT(DISTINCT orderkey) AS orders
FROM dev.gold.fact_sales
GROUP BY l_returnflag
ORDER BY net_revenue DESC;

In [0]:
-- Q15: Average Discount by Market Segment
SELECT dc.market_segment,
    AVG(fs.discount) AS avg_discount,
    SUM(fs.extendedprice * (1 - fs.discount)) AS net_revenue
FROM dev.gold.fact_sales fs
JOIN dev.gold.dim_customer dc ON fs.customerkey = dc.customerkey
GROUP BY dc.market_segment
ORDER BY avg_discount DESC;

In [0]:
-- Q16: Average Ship-to-Receipt Days by Shipping Mode
SELECT l_linestatus AS shipping_mode,
    AVG(DATEDIFF(l_receiptdate, l_shipdate)) AS avg_ship_to_receipt_days,
    COUNT(DISTINCT orderkey) AS orders
FROM dev.gold.fact_sales
GROUP BY l_linestatus
ORDER BY avg_ship_to_receipt_days DESC;

In [0]:
-- Q17: Order Count and Revenue by Order Priority
SELECT o_orderpriority,
    COUNT(DISTINCT orderkey) AS orders,
    SUM(extendedprice * (1 - discount)) AS net_revenue
FROM dev.gold.fact_sales
GROUP BY o_orderpriority
ORDER BY net_revenue DESC;

In [0]:
-- Q18: Monthly Breakdown of Order Status
SELECT YEAR(orderdate) AS year,
    MONTH(orderdate) AS month,
    o_orderstatus,
    COUNT(DISTINCT orderkey) AS orders
FROM dev.gold.fact_sales
GROUP BY year, month, o_orderstatus
ORDER BY year, month, o_orderstatus;

In [0]:
-- Q19: Revenue per Unit Quantity by Product (Top 10)
SELECT dp.product_name,
    SUM(fs.extendedprice * (1 - fs.discount)) / SUM(fs.quantity) AS revenue_per_unit,
    SUM(fs.quantity) AS total_quantity
FROM dev.gold.fact_sales fs
JOIN dev.gold.dim_product dp ON fs.productkey = dp.productkey
GROUP BY dp.product_name
ORDER BY revenue_per_unit DESC
LIMIT 10;

In [0]:
-- Q20: Delivered Orders by Commitment vs. Receive Date
SELECT
    CASE WHEN l_commitdate = l_receiptdate THEN 'On Time' ELSE 'Late' END AS delivery_status,
    COUNT(DISTINCT orderkey) AS orders,
    SUM(extendedprice * (1 - discount)) AS net_revenue
FROM dev.gold.fact_sales
GROUP BY delivery_status
ORDER BY net_revenue DESC;

# Final Business Challenge Summary

**How is our business performing, and where should management focus its attention?**

## Key Findings
1. **Strong overall revenue and customer base**: The business generated nearly $1.09T in net revenue with 7.5M orders and ~500K active customers (Cell 1).
2. **Top customers and products concentrate value**: The top 10 customers and top 10 products each account for significant shares of total revenue (Cells 5 & 7).
3. **Furniture and Building segments drive the most revenue**: These market segments lead as contributors, each exceeding $218B net revenue (Cell 4).

## Opportunities for Growth
1. **Expand into underperforming segments or regions**: Revenue is concentrated in key markets and segments; Household and Automobile segments and regions such as America/Africa have lower revenue (Cells 4 & 12).
2. **Discount optimization**: Average discount varies by segment; optimizing discount policies may drive higher profitability (Cell 15).

## Risks / Investigation Areas
1. **Supplier concentration risk**: The top 10 suppliers account for a large share (0.03% of total) of revenue, posing dependency risk (Cells 10 & 11).
2. **Shipping mode delivery variance**: Some shipping modes show longer average ship-to-receipt days, potentially impacting customer satisfaction (Cell 16).

## Recommended Management Actions
- Invest in broadening market segment and regional reach.
- Review discount and pricing strategy by segment.
- Monitor supplier dependency and diversify supply chain as needed.
- Investigate disparities in shipping mode delivery times and consider process improvements.